# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library. All data schema elements (record sets, fields, etc.) are referenced by their Croissant `@id`.

### Dataset Source
The Croissant schema for this dataset is available at:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and record information from the FAIR^2 dataset using `mlcroissant`. 

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Explore available record sets and fields by `@id`. Croissant schemas reference each entity by these unique IRIs.

In [ ]:
# List available record sets by @id
record_sets = list(dataset.record_sets)
if not record_sets:
    raise ValueError("No record_sets found in the Croissant schema!")
print("Record Sets (@id):")
for rs in record_sets:
    print(f"- {rs['@id']}")
    # List fields for each record set
    if 'field' in rs:
        print("  Fields:")
        for fld in rs['field']:
            print(f"    - {fld['@id']}: {fld.get('name', '')} (dataType: {fld.get('dataType','')})")

## 3. Data Extraction

Load data from each record set into a pandas DataFrame for analysis. Each record set and field is referenced by its `@id`.

For this dataset, the main record set is likely to be a table named like `clinical_table` or similar. We'll load all detected record sets.

In [ ]:
# Build list of record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
# For demonstration, print how many record sets discovered
print(f"Found {len(record_set_ids)} record sets:")
print('\n'.join(record_set_ids))
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"No records found for {record_set_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {df.shape[0]} records for record set {record_set_id}")
# For further steps, select the main tabular record set (the one with most records/columns)
main_record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[1])
print(f"\nMain record set selected for EDA: {main_record_set_id}\n")
print("Columns available:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply filtering, normalization, and simple grouping operations. All variables are referenced by their `@id`, as per Croissant conventions.

We'll automatically choose the first numeric field (e.g., Age, or another continuous variable) and the first categorical field available, using their `@id` as column name.

In [ ]:
import numpy as np
# Automatically detect a numeric field by @id
df = dataframes[main_record_set_id]
numeric_field = None
for col in df.columns:
    if np.issubdtype(df[col].dropna().dtype, np.number):
        numeric_field = col
        break
if numeric_field is None:
    # Try to convert likely columns to numeric
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if np.issubdtype(df[col].dropna().dtype, np.number):
                numeric_field = col
                break
        except Exception:
            continue
if numeric_field is None:
    raise ValueError("No numeric fields found in the main record set for EDA.")
print(f"Selected numeric field (by @id): {numeric_field}")

# Pick a threshold for filtering, use median+std as example if >0, otherwise 10
series = df[numeric_field].dropna()
threshold = float(series.median()+series.std()) if series.std() > 0 else 10
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df[[numeric_field]].head())

# Normalize field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Use first categorical/text field as group key (excluding the numeric one)
group_field = None
for col in df.columns:
    if col != numeric_field and (df[col].dtype == 'object' or df[col].dtype.name == 'category'):
        group_field = col
        break
if group_field:
    # Group and show the mean of numeric field
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped filtered data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())
else:
    print('No suitable categorical/text group field found for grouping.')

## 5. Visualization

Below are example visualizations of the numeric field distribution and the mean per group, referencing columns by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), bins=15, kde=True, color='dodgerblue')
plt.xlabel(f'{numeric_field} (@id)')
plt.title(f'Distribution of {numeric_field} (@id)')
plt.show()

# If grouping, bar plot of group means
if group_field:
    top_groups = grouped_df.sort_values(by=numeric_field, ascending=False).head(10)
    plt.figure(figsize=(8,4))
    sns.barplot(x=group_field, y=numeric_field, data=top_groups, palette='Set2')
    plt.xlabel(f'{group_field} (@id)')
    plt.ylabel(f'Mean {numeric_field}')
    plt.title(f'Mean {numeric_field} per {group_field}')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- This notebook demonstrates reading, filtering, normalization, and grouping in the FAIR^2 clinical oncology dataset using `mlcroissant`. All operations were done referencing Croissant `@id` identifiers for full traceability.
- The dataset provides rich demographic, clinical, and molecular information for research on second primary colorectal cancer in cancer survivors, with fields covering a range of numeric and categorical variables.
- Further analysis is encouraged, focusing on relationships between anatomical location, biomarker status, and clinical outcomes, always referencing entities via `@id` for clarity and reproducibility.
